In [28]:
from src.utils import *
from src.data import eICUData 
from src.parser import eICUConfigParser
from src.helpers import *

parser = eICUConfigParser('simplified')
datapath = 'simplified'

reload_data = False

if reload_data:
    eicu, processed_dt, ncm = get_eicu_data_bundle(parser, hospital_filter=get_hospitals_with('all'))
    eicu.save(datapath)
else:
    eicu, processed_dt, ncm = get_eicu_data_bundle(
        parser, 
        datapath=datapath,
        u_size={('age', 'ethnicity', 'gender'): 3}
    )

processed_dt.print_df()

/Users/Hanita/Projects/COMS4995:BINF4008-ML4HealthMed/ehr-ncm/src/data/utils.py:55: UserWarning: The following features were not assigned to any variable: {'uniquepid', 'hospitalid'}
  warnings.warn('The following features were not assigned to any variable: {}'.format(unassigned_features), UserWarning)


It is okay to exclude features from the model but they will not be used in the causal analysis.


,ethnicity,age,gender,sao2,heartrate,respiration,bp,sepsis,ventilation,intravenous_fluid,mortality
0,0.0,1.0,0.0,1.0,0.5,0.5,0.0,1.0,1.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
130680,0.0,0.0,1.0,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0
130681,0.0,1.0,1.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0
130682,1.0,1.0,1.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0
130683,1.0,1.0,1.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0


In [29]:
from src.train import *
from src.utils import *
from src.data import eICUData 
from src.parser import eICUConfigParser
from src.helpers import *

# eicu, processed_dt, ncm = get_eicu_data_bundle('simplified', datapath='simplified')
retrain_ncm = True
train_df, test_df, train_dataloader, test_dataloader = processed_dt.train_test_split(
    train_hospitals='all'
)

if retrain_ncm:
    _ = train_ncm(ncm, train_dataloader, hyperparameters={'n-epochs': 3})
    ncm.save('simplified_allhosp_3epoch')

compute_accuracy(
    ncm,
    test_dataloader,
    "sepsis"
)

Train epoch 1: 100%|██████████| 3673/3673 [02:21<00:00, 26.03it/s, loss=0.0199] 


Epoch 1/3, Loss: 0.0802


Train epoch 2: 100%|██████████| 3673/3673 [02:24<00:00, 25.50it/s, loss=0.0249] 


Epoch 2/3, Loss: 0.0713


Train epoch 3: 100%|██████████| 3673/3673 [02:24<00:00, 25.38it/s, loss=0.0299] 


Epoch 3/3, Loss: 0.0709
Model saved to simplified_allhosp_3epoch


Computing  accuracy: 100%|██████████| 412/412 [00:11<00:00, 34.36it/s]

		 energy-based: 0.9831
		 js-divergence: 0.9686


tensor(0.9335, device='mps:0')

In [3]:
for ft in eicu.config.assignments:
    print(compute_accuracy(ncm, test_dataloader, ft, label=ft), '\n')

Computing ethnicity accuracy: 100%|██████████| 412/412 [00:11<00:00, 36.49it/s]


		 energy-based: 0.9739
		 js-divergence: 0.9914
tensor(0.9314, device='mps:0') 



Computing age accuracy: 100%|██████████| 412/412 [00:11<00:00, 36.84it/s]


		 energy-based: 0.9658
		 js-divergence: 0.9833
tensor(0.9338, device='mps:0') 



Computing gender accuracy: 100%|██████████| 412/412 [00:11<00:00, 37.24it/s]


...

		 energy-based: 0.9841
		 js-divergence: 0.9704
tensor(0.9346, device='mps:0') 



Computing sep_tr accuracy: 100%|██████████| 412/412 [00:12<00:00, 34.25it/s]


		 energy-based: 0.9815
		 js-divergence: 0.9467
tensor(0.9307, device='mps:0') 



Computing mortality accuracy: 100%|██████████| 412/412 [00:11<00:00, 34.53it/s]

		 energy-based: 0.9911
		 js-divergence: 0.9790
tensor(0.9349, device='mps:0') 



In [52]:
sepsis = processed_dt.data[processed_dt.data['sepsis']==1][processed_dt.data['mortality']==1]
sepsis 

/var/folders/gq/nr1qyp1s0f76pz81bn_6vlzc0000gq/T/ipykernel_79830/1070738704.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  sepsis = processed_dt.data[processed_dt.data['sepsis']==1][processed_dt.data['mortality']==1]


,uniquepid,hospitalid,age,gender,ethnicity,mortality,sepsis,ventilation,intravenous_fluid,bp,sao2,heartrate,respiration
71,003-11513,79,0.25,0.5,0.4,1.0,1.0,1.0,0.0,1.000000,0.666667,0.5,0.00
82,003-1168,79,0.25,0.5,0.4,1.0,1.0,1.0,1.0,0.333333,0.333333,0.0,0.50
96,003-12079,79,0.25,0.0,0.4,1.0,1.0,1.0,1.0,0.333333,0.333333,0.5,0.00
143,003-13132,79,0.75,0.5,0.8,1.0,1.0,1.0,0.0,0.333333,0.666667,0.5,0.00
181,003-13962,84,0.50,0.5,1.0,1.0,1.0,1.0,1.0,1.000000,0.333333,0.5,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...
130265,035-7324,459,0.50,0.0,1.0,1.0,1.0,1.0,0.0,0.333333,0.333333,0.0,0.50
130502,035-8898,458,0.50,0.5,0.4,1.0,1.0,1.0,1.0,0.333333,0.666667,0.0,0.50
130549,035-9139,458,0.50,0.5,0.0,1.0,1.0,1.0,1.0,0.333333,0.333333,0.0,0.50
130556,035-9181,458,0.25,0.5,0.8,1.0,1.0,1.0,0.0,0.333333,0.333333,0.0,0.25


In [53]:
sepsis.loc[71].to_dict()

{'uniquepid': '003-11513',
 'hospitalid': 79,
 'age': 0.25,
 'gender': 0.5,
 'ethnicity': 0.4,
 'mortality': 1.0,
 'sepsis': 1.0,
 'ventilation': 1.0,
 'intravenous_fluid': 0.0,
 'bp': 1.0,
 'sao2': 0.6666666666666666,
 'heartrate': 0.5,
 'respiration': 0.0}

In [54]:
import torch as T
from src.metric import * 
from src.metric.queries import _get_conditioned_u
ev={
    'age': 0.25,
    'gender': 0.5,
    'ethnicity': 0.4,
    'mortality': 1.0,
    'sepsis': 1.0,
    'ventilation': 1.0,
    'intravenous_fluid': 0.0,
    'bp': 1.0,
    'sao2': 0.6666666666666666,
    'heartrate': 0.5,
    'respiration': 0.0
}

evidence = {
    'ethnicity': 0.4,
    'age': 0.5,
    'gender': 0.0,
    'vitals': T.tensor([0.3333333333333333, 0.0, 0.5, 0.3333333333333333]),
}

# probability(ncm, "sepsis", 1,
#     evidence=evidence
# )

samples = ncm(n=10000, evaluating=True)
samples.keys()


KeyboardInterrupt: 

In [55]:
ev={
    'age': 0.25,
    'gender': 0.5,
    'ethnicity': 0.4,
    'mortality': 1.0,
    'sepsis': 1.0,
    'ventilation': 1.0,
    'intravenous_fluid': 0.0,
    'bp': 1.0,
    'sao2': 0.6666666666666666,
    'heartrate': 0.5,
    'respiration': 0.0
}
ev = {k: processed_dt.scale[k](T.tensor([[v]])) for k, v in ev.items()}
{k: T.cat([ev.get(ft, T.tensor([[]])).flatten() for ft in v]) for k,v in processed_dt.assignments.items()}

{'ethnicity': tensor([2.]),
 'age': tensor([1.]),
 'gender': tensor([1.]),
 'vitals': tensor([2., 1., 0., 3.]),
 'sepsis': tensor([1.]),
 'sep_tr': tensor([1., 0.]),
 'mortality': tensor([1.])}

In [61]:
(
    processed_dt.data
    # [(processed_dt.data['sepsis']==1) & processed_dt.data['mortality']==1]
    .groupby(["age", "gender", "ethnicity"])
    [["sepsis", "mortality"]]
    # .agg(
    #     sepsis=("sepsis", "count"),
    #     mortality=("mortality", "count")
    # )
    .describe()
)

sepsis                                               \
                        count      mean       std  min  25%  50%  75%  max   
age gender ethnicity                                                         
0.0 0.0    0.0        29777.0  0.159486  0.366134  0.0  0.0  0.0  0.0  1.0   
           1.0         7296.0  0.163788  0.370109  0.0  0.0  0.0  0.0  1.0   
    1.0    0.0        33725.0  0.151875  0.358905  0.0  0.0  0.0  0.0  1.0   
           1.0         7703.0  0.170843  0.376396  0.0  0.0  0.0  0.0  1.0   
1.0 0.0    0.0        16096.0  0.141588  0.348638  0.0  0.0  0.0  0.0  1.0   
           1.0         6768.0  0.135638  0.342429  0.0  0.0  0.0  0.0  1.0   
    1.0    0.0        20726.0  0.115748  0.319931  0.0  0.0  0.0  0.0  1.0   
           1.0         8594.0  0.114615  0.318575  0.0  0.0  0.0  0.0  1.0   

                     mortality                                               
                         count      mean       std  min  25%  50%  75%  max  
age gender ethnicity                                                         
0.0 0.0    0.0         29777.0  0.094502  0.292531  0.0  0.0  0.0  0.0  1.0  
           1.0          7296.0  0.092928  0.290351  0.0  0.0  0.0  0.0  1.0  
    1.0    0.0         33725.0  0.094589  0.292650  0.0  0.0  0.0  0.0  1.0  
           1.0          7703.0  0.101389  0.301863  0.0  0.0  0.0  0.0  1.0  
1.0 0.0    0.0         16096.0  0.048956  0.215783  0.0  0.0  0.0  0.0  1.0  
           1.0          6768.0  0.052157  0.222360  0.0  0.0  0.0  0.0  1.0  
    1.0    0.0         20726.0  0.054859  0.227710  0.0  0.0  0.0  0.0  1.0  
           1.0          8594.0  0.056435  0.230773  0.0  0.0  0.0  0.0  1.0

In [63]:
data_age_cut = eicu.data.copy()
data_age_cut['age'] = pd.cut(eicu.data['age'], bins=[-1, 60, 1000], labels=["young", "old"])
(
    data_age_cut
    .groupby(["age", "gender", "ethnicity"])
    [["sepsis", "mortality"]]
    .describe()
)

sepsis                                               \
                          count      mean       std  min  25%  50%  75%  max   
age   gender ethnicity                                                         
young Female Caucasian  16096.0  0.141588  0.348638  0.0  0.0  0.0  0.0  1.0   
             POC/Other   6768.0  0.135638  0.342429  0.0  0.0  0.0  0.0  1.0   
      Male   Caucasian  20726.0  0.115748  0.319931  0.0  0.0  0.0  0.0  1.0   
             POC/Other   8594.0  0.114615  0.318575  0.0  0.0  0.0  0.0  1.0   
old   Female Caucasian  29777.0  0.159486  0.366134  0.0  0.0  0.0  0.0  1.0   
             POC/Other   7296.0  0.163788  0.370109  0.0  0.0  0.0  0.0  1.0   
      Male   Caucasian  33725.0  0.151875  0.358905  0.0  0.0  0.0  0.0  1.0   
             POC/Other   7703.0  0.170843  0.376396  0.0  0.0  0.0  0.0  1.0   

                       mortality                                               
                           count      mean       std  min  25%  50%  75%  max  
age   gender ethnicity                                                         
young Female Caucasian   16095.0  0.048959  0.215790  0.0  0.0  0.0  0.0  1.0  
             POC/Other    6768.0  0.052157  0.222360  0.0  0.0  0.0  0.0  1.0  
      Male   Caucasian   20726.0  0.054859  0.227710  0.0  0.0  0.0  0.0  1.0  
             POC/Other    8594.0  0.056435  0.230773  0.0  0.0  0.0  0.0  1.0  
old   Female Caucasian   29775.0  0.094509  0.292540  0.0  0.0  0.0  0.0  1.0  
             POC/Other    7296.0  0.092928  0.290351  0.0  0.0  0.0  0.0  1.0  
      Male   Caucasian   33724.0  0.094591  0.292654  0.0  0.0  0.0  0.0  1.0  
             POC/Other    7703.0  0.101389  0.301863  0.0  0.0  0.0  0.0  1.0

In [64]:
import torch as T
from src.data import * 
from src.utils import *
from src.metric.counterfactual import *
from src.metric import check_equal

# eicu, processed_dt, ncm = get_eicu_data_bundle('simplified', datapath='simplified', modelpath='simplified_allhosp_3epoch')

evidence = {
    'ethnicity': T.tensor([1.]),
    'age': T.tensor([0.]),
    'gender': T.tensor([1.]),
    # 'vitals': T.tensor([2., 1., 0., 3.]),
    # 'sepsis': T.tensor([1.]),
    # 'sep_tr': T.tensor([1., 0.]),
    # 'mortality': T.tensor([1.])
}

patient = CTFTerm(evidence.keys(), {}, evidence)
sepsis = CTFTerm({"sepsis"}, {}, {"sepsis": 1})

n_new = 100000
u_ORIG = ncm.pu.sample(n_new)
u = u_ORIG

psepsis_patient = CTF({sepsis}, {patient}, name="Probability")
samples = ncm(n=None, u=u, do={}, evaluating=True)

cond_match = T.ones(n_new, dtype=T.bool)
for (k,v) in patient.var_vals.items():
    cond_match *= check_equal(samples[k], v)

u = {k: v[cond_match] for (k, v) in u.items()}
n_new = T.sum(cond_match.long()).item()

In [91]:
def sample_to_df(sample_dict):
    samples_df = pd.DataFrame({k: v.tolist() for k,v in sample_dict.items()})
    for ft in ['gender', 'age', 'ethnicity', 'sepsis', 'mortality']:
        samples_df[ft] = samples_df[ft].map(lambda x: x[0])
    return samples_df

samples_df = sample_to_df(samples)
samples_df

,gender,age,ethnicity,vitals,sepsis,sep_tr,mortality
0,0.0,1.0,1.0,"[0.0, 0.0, 1.0, 2.0]",0.0,"[1.0, 0.0]",0.0
1,1.0,1.0,1.0,"[0.0, 2.0, 0.0, 0.0]",1.0,"[0.0, 1.0]",0.0
2,1.0,0.0,0.0,"[0.0, 0.0, 1.0, 0.0]",0.0,"[0.0, 0.0]",0.0
3,1.0,0.0,0.0,"[1.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",0.0
4,1.0,1.0,0.0,"[0.0, 2.0, 0.0, 2.0]",0.0,"[0.0, 0.0]",0.0
...,...,...,...,...,...,...,...
99995,1.0,0.0,0.0,"[0.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",0.0
99996,1.0,0.0,0.0,"[1.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",1.0
99997,0.0,0.0,0.0,"[0.0, 0.0, 1.0, 0.0]",0.0,"[0.0, 0.0]",0.0
99998,0.0,1.0,0.0,"[0.0, 0.0, 2.0, 0.0]",0.0,"[0.0, 0.0]",0.0


In [67]:
samples_df[samples_df['sepsis']==1][['age', 'gender', 'ethnicity']].value_counts()

age  gender  ethnicity
1.0  1.0     0.0          2303
0.0  1.0     0.0          2157
     0.0     1.0          1936
1.0  0.0     1.0          1215
0.0  1.0     1.0          1109
1.0  0.0     0.0          1027
     1.0     1.0           766
0.0  0.0     0.0           610
Name: count, dtype: int64

In [68]:
samples_df[['age', 'gender', 'ethnicity']].value_counts()

age  gender  ethnicity
0.0  1.0     0.0          21373
     0.0     0.0          17420
1.0  1.0     0.0          14853
     0.0     0.0          10776
0.0  1.0     1.0          10173
     0.0     1.0          10168
1.0  0.0     1.0           7899
     1.0     1.0           7338
Name: count, dtype: int64

In [71]:
match = T.ones(n_new, dtype=T.bool, requires_grad=False) 
q_samples = ncm(n=None, u=u, do={}, evaluating=True)

{'gender': tensor([[1.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]]),
 'age': tensor([[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]),
 'ethnicity': tensor([[1.],
         [1.],
         [1.],
         ...,
         [1.],
         [1.],
         [1.]]),
 'vitals': tensor([[1., 2., 1., 0.],
         [0., 0., 2., 0.],
         [0., 0., 0., 0.],
         ...,
         [0., 0., 0., 0.],
         [1., 0., 0., 0.],
         [1., 0., 0., 0.]]),
 'sepsis': tensor([[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]),
 'sep_tr': tensor([[1., 0.],
         [0., 0.],
         [0., 0.],
         ...,
         [0., 0.],
         [0., 0.],
         [0., 0.]]),
 'mortality': tensor([[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [1.],
         [0.]])}

In [96]:
# pd.DataFrame(q_samples)
q_samples_df = sample_to_df(q_samples)
q_samples_df

,gender,age,ethnicity,vitals,sepsis,sep_tr,mortality
0,1.0,0.0,1.0,"[1.0, 2.0, 1.0, 0.0]",0.0,"[1.0, 0.0]",0.0
1,1.0,0.0,1.0,"[0.0, 0.0, 2.0, 0.0]",0.0,"[0.0, 0.0]",0.0
2,1.0,0.0,1.0,"[0.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",0.0
3,1.0,0.0,1.0,"[0.0, 1.0, 0.0, 2.0]",0.0,"[0.0, 0.0]",0.0
4,1.0,0.0,1.0,"[1.0, 1.0, 0.0, 0.0]",1.0,"[1.0, 0.0]",1.0
...,...,...,...,...,...,...,...
10168,1.0,0.0,1.0,"[0.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",0.0
10169,1.0,0.0,1.0,"[1.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",1.0
10170,1.0,0.0,1.0,"[0.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",0.0
10171,1.0,0.0,1.0,"[1.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0]",1.0


In [97]:
match *= check_equal(q_samples['sepsis'], 1)
(T.sum(match.long()) / match.shape[0]).item()

0.10901405662298203

In [108]:
class CTFTerm:
    def __init__(self, vars: Iterable, do_vals: dict={}, var_vals: dict = None):
        self.vars = set(to_iter(vars))
        self.do_vals = do_vals
        if var_vals is None:
            self.var_vals = dict()
        else:
            assert set(var_vals.keys()).issubset(self.vars)
            self.var_vals = var_vals

        self.val_match = (self.vars == set(self.var_vals.keys()))

    def has_all_values(self):
        return self.val_match

    def is_degenerate(self):
        return len(self.vars) == 0

    def strip_values(self):
        return CTFTerm(self.vars, self.do_vals, {})
    
    def __str__(self):
        out = "["
        for i, var in enumerate(self.vars):
            if var in self.var_vals:
                out += "{} = {}".format(var, self.var_vals[var])
            else:
                out += var
            if i != len(self.vars) - 1:
                out += ", "
        if len(self.do_vals) > 0:
            out += " | do("
            for i, var in enumerate(self.do_vals):
                out += "{} = {}".format(var, self.do_vals[var])
                if i != len(self.do_vals) - 1:
                    out += ", "
            out += ")"
        out += "]"
        return out

    def __eq__(self, other):
        if not isinstance(other, CTFTerm):
            return False
        return self.vars == other.vars and self.do_vals == other.do_vals and self.var_vals == other.var_vals

    def __hash__(self):
        return hash((frozenset(self.vars), frozenset(self.do_vals.items()), frozenset(self.var_vals.items())))

    

class CTF:
    def __init__(self, term_set: set = None, cond_term_set: set = None, name=None):
        if term_set is None:
            self.term_set = set()
        else:
            degenerate_terms = set()
            for term in term_set:
                assert isinstance(term, CTFTerm)
                if term.is_degenerate():
                    degenerate_terms.add(term)
            self.term_set = term_set.difference(degenerate_terms)

        if cond_term_set is None:
            self.cond_term_set = set()
        else:
            degenerate_terms = set()
            for term in cond_term_set:
                assert isinstance(term, CTFTerm)
                assert term.has_all_values()
                if term.is_degenerate():
                    degenerate_terms.add(term)
            self.cond_term_set = cond_term_set.difference(degenerate_terms)

        self.name = name

    def add_term(self, term: CTFTerm):
        if not term.is_degenerate():
            self.term_set.add(term)

    def add_cond_term(self, term: CTFTerm):
        assert term.has_all_values()
        if not term.is_degenerate():
            self.cond_term_set.add(term)

    def get_vars(self):
        var_set = set()
        for term in self.term_set:
            var_set = var_set.union(term.vars)
        return var_set

    def get_cond_vars(self):
        var_set = set()
        for term in self.cond_term_set:
            var_set = var_set.union(term.vars)
        return var_set

    def has_all_values(self):
        for term in self.term_set:
            if not term.has_all_values():
                return False
        return True

    def search_value(self, q_var, q_do_vals):
        do_vars = set(q_do_vals.keys())
        for term in self.term_set:
            if q_var in term.var_vals and do_vars == set(term.do_vals.keys()):
                return term.var_vals[q_var]
        return None

    def is_degenerate(self):
        return len(self.term_set) == 0

    def get_full_joint(self):
        return CTF(self.term_set.union(self.cond_term_set))

    def drop_cond_ctf(self):
        return CTF(self.term_set, set())

    def get_cond_ctf(self):
        return CTF(self.cond_term_set, set())

    def strip_values(self):
        new_term_set = set()
        for term in self.term_set:
            new_term_set.add(term.strip_values())

        new_cond_term_set = set()
        for term in self.cond_term_set:
            new_cond_term_set.add(term.strip_values())

        return CTF(new_term_set, new_cond_term_set)

    def __str__(self):
        out = "P("
        for i, term in enumerate(self.term_set):
            out += str(term)
            if i != len(self.term_set) - 1:
                out += ", "
        if len(self.cond_term_set) > 0:
            out += " | "
            for i, term in enumerate(self.cond_term_set):
                out += str(term)
                if i != len(self.cond_term_set) - 1:
                    out += ", "
        out += ")"
        return out


In [107]:
from src.metric.queries import _get_conditioned_u
    
def _probability(model, var, val, evidence={}, intervention={}, neq_evidence={}, u=None):
    """ This will calculate the probability that a variable equals a certain value given the dictionary of evidence, and the dictionary of interventions. Both dictionaries are optional.
    - `probability('Y',1,{'X':0},{'Z':1})` will return $P(Y=1 | X=0, do(Z=1))$
    - `probability('Y',1,intervention={'Z':1})` will return $P(Y=1 | do(Z=1))$
    """
    U, _ = _get_conditioned_u(model, u, conditions=evidence, opposite_conditions=neq_evidence, n=100000)
    sampleY = sample_ctf(model, CTFTerm(var, do_vals=intervention), u=U)[var]
    n = sampleY.numel()
    return 0 if n==0 else (sampleY == val).sum().item() / n 

def probability(model, var, val, evidence={}, intervention={}):
    """ This will calculate the probability that a variable equals a certain value given the dictionary of evidence, and the dictionary of interventions. Both dictionaries are optional.
    - `probability('Y',1,{'X':0},{'Z':1})` will return $P(Y=1 | X=0, do(Z=1))$
    - `probability('Y',1,intervention={'Z':1})` will return $P(Y=1 | do(Z=1))$
    """
    return _probability(model, var, val, evidence=evidence, intervention=intervention) 



probability(ncm, "sepsis", 1, {'ethnicity': 1, 'age': 0, 'gender': 1})

0.11640108822386319

In [82]:
(
    train_df
    .groupby(["age", "gender", "ethnicity"])
    [["sepsis", "mortality"]]
    .describe()
)

sepsis                                               \
                        count      mean       std  min  25%  50%  75%  max   
age gender ethnicity                                                         
0.0 0.0    0.0        26802.0  0.159279  0.365943  0.0  0.0  0.0  0.0  1.0   
           1.0         6564.0  0.163924  0.370235  0.0  0.0  0.0  0.0  1.0   
    1.0    0.0        30358.0  0.150438  0.357506  0.0  0.0  0.0  0.0  1.0   
           1.0         6907.0  0.168814  0.374615  0.0  0.0  0.0  0.0  1.0   
1.0 0.0    0.0        14421.0  0.141252  0.348294  0.0  0.0  0.0  0.0  1.0   
           1.0         6105.0  0.136282  0.343116  0.0  0.0  0.0  0.0  1.0   
    1.0    0.0        18646.0  0.116540  0.320880  0.0  0.0  0.0  0.0  1.0   
           1.0         7716.0  0.114178  0.318048  0.0  0.0  0.0  0.0  1.0   

                     mortality                                               
                         count      mean       std  min  25%  50%  75%  max  
age gender ethnicity                                                         
0.0 0.0    0.0         26802.0  0.094172  0.292073  0.0  0.0  0.0  0.0  1.0  
           1.0          6564.0  0.092169  0.289287  0.0  0.0  0.0  0.0  1.0  
    1.0    0.0         30358.0  0.094571  0.292627  0.0  0.0  0.0  0.0  1.0  
           1.0          6907.0  0.101491  0.302000  0.0  0.0  0.0  0.0  1.0  
1.0 0.0    0.0         14421.0  0.049303  0.216508  0.0  0.0  0.0  0.0  1.0  
           1.0          6105.0  0.053071  0.224194  0.0  0.0  0.0  0.0  1.0  
    1.0    0.0         18646.0  0.055132  0.228245  0.0  0.0  0.0  0.0  1.0  
           1.0          7716.0  0.055728  0.229411  0.0  0.0  0.0  0.0  1.0

In [106]:
def print_sepsis_mortality_totals(df, label=None):
    if label: print(f"--{label}--")
    print(f"Total: {len(df)}")
    print(f"Sepsis: {sum(df['sepsis'])} ({sum(df['sepsis'])*100/len(df):.2f}%)")
    print(f"Mortality: {sum(df['mortality'])} ({sum(df['mortality'])*100/len(df):.2f}%)\n")

train_demo_df = train_df[(train_df['age']==0) & (train_df['gender']==1) & (train_df['ethnicity']==1)]
test_demo_df = test_df[(test_df['age']==0) & (test_df['gender']==1) & (test_df['ethnicity']==1)]
total_demo_df = processed_dt.data[(processed_dt.data['age']==0) & (processed_dt.data['gender']==1) & (processed_dt.data['ethnicity']==1)]

print_sepsis_mortality_totals(train_demo_df, 'Train DF')
print_sepsis_mortality_totals(test_demo_df, 'Test DF')
print_sepsis_mortality_totals(total_demo_df, 'Total Processed DF')
print_sepsis_mortality_totals(sample_to_df(samples), 'Unfiltered Samples')
# print_sepsis_mortality_totals(sample_to_df({k: v[cond_match] for (k, v) in samples.items()}), 'Filtered Samples')
print_sepsis_mortality_totals(q_samples_df, 'Generated from Samples')

--Train DF--
Total: 6907
Sepsis: 1166.0 (16.88%)
Mortality: 701.0 (10.15%)

--Test DF--
Total: 796
Sepsis: 150.0 (18.84%)
Mortality: 80.0 (10.05%)

--Total Processed DF--
Total: 7703
Sepsis: 1316.0 (17.08%)
Mortality: 781.0 (10.14%)

--Unfiltered Samples--
Total: 100000
Sepsis: 11123.0 (11.12%)
Mortality: 9587.0 (9.59%)

--Generated from Samples--
Total: 10173
Sepsis: 1109.0 (10.90%)
Mortality: 1850.0 (18.19%)



In [123]:
def get_distribution_diffs(model, dataloader, target_var, device=DEVICE, label=""):
    model.eval()
    total_loss = 0
    total_e = 0
    total_js = 0
    with T.no_grad():
        pbar = tqdm(dataloader, desc=f"Computing {label} accuracy")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            batch_size = next(iter(batch.values())).shape[0]

            preds=model(n=batch_size,select={target_var})[target_var]
            # labels = batch[target_var]
            
            labels = T.cat([batch[target_var]], axis=1)
            pred_labels = T.cat([preds], axis=1)
            total_loss += MMD_loss(labels,pred_labels)

            total_e += energy_distance(labels.cpu(),pred_labels.cpu())
            total_js += classifier_js_divergence(labels.cpu(),pred_labels.cpu())

    return total_e/len(dataloader), total_js/len(dataloader), total_loss/len(dataloader)

print(f"Sepsis: {get_distribution_diffs(ncm, test_dataloader, 'sepsis', label='sepsis')}")
print(f"Mortality: {get_distribution_diffs(ncm, test_dataloader, 'mortality', label='mortality')}")

Computing sepsis accuracy: 100%|██████████| 412/412 [00:25<00:00, 15.98it/s]


Sepsis: (0.01587788709424726, 0.029590726602319053, tensor(0.0657, device='mps:0'))


Computing mortality accuracy: 100%|██████████| 412/412 [00:26<00:00, 15.32it/s]

Mortality: (0.009837568577579177, 0.022253394673001286, tensor(0.0648, device='mps:0'))


In [122]:
6.899482874243386/len(test_dataloader)

0.01674631765593055

In [110]:
def compute_ctf(model, query: CTF, n=1000000, u=None, get_prob=True, evaluating=False):
    if u is None:
        u = model.pu.sample(n)
        n_new = n
    else:
        n_new = len(u[next(iter(u))])

    for term in query.cond_term_set:
        samples = model(n=None, u=u, do={
            k: expand_do(v, n_new) for (k, v) in term.do_vals.items()
        }, select=term.vars, evaluating=True)

        cond_match = T.ones(n_new, dtype=T.bool)
        for (k, v) in term.var_vals.items():
            cond_match *= check_equal(samples[k], v)

        u = {k: v[cond_match] for (k, v) in u.items()}
        n_new = T.sum(cond_match.long()).item()

    if n_new <= 0:
        if evaluating:
            return float('nan')
        else:
            return T.tensor([float('nan')]).to(model.device_param)

    if evaluating:
        match = T.ones(n_new, dtype=T.bool, requires_grad=False)
        out_samples = dict()
        for term in query.term_set:
            expanded_do_terms = dict()
            for (k, v) in term.do_vals.items():
                if k == "nested":
                    expanded_do_terms.update(model.compute_ctf(v, u=u, get_prob=False, evaluating=evaluating))
                else:
                    expanded_do_terms[k] = expand_do(v, n_new)
            q_samples = model(n=None, u=u, do=expanded_do_terms, select=term.vars, evaluating=evaluating)

            if get_prob:
                for (k, v) in term.var_vals.items():
                    match *= check_equal(q_samples[k], v)
            else:
                out_samples.update(q_samples)

        if get_prob:
            return (T.sum(match.long()) / match.shape[0]).item()
        else:
            return out_samples
    
    else:
        loss = 0
        loss_count = 0
        out_samples = dict()
        for term in query.term_set:
            expanded_do_terms = dict()
            for (k, v) in term.do_vals.items():
                if k == "nested":
                    expanded_do_terms.update(model.compute_ctf(v, u=u, get_prob=False, evaluating=evaluating))
                else:
                    expanded_do_terms[k] = expand_do(v, n_new)

            q_samples = model(n=None, u=u, do=expanded_do_terms, select=term.vars, evaluating=evaluating)

            if get_prob:
                for (k, v) in term.var_vals.items():
                    loss += model.query_loss(q_samples[k], v)
                    loss_count += 1
            else:
                out_samples.update(q_samples)

        if get_prob:
            return loss / (n_new * loss_count)
        else:
            return out_samples


In [112]:
patient = CTFTerm(['age', 'gender', 'ethnicity'], {}, {'age':0, 'gender':1, 'ethnicity':1})
sepsis = CTFTerm({"sepsis"}, {}, {"sepsis": 1})
dead = CTFTerm({"mortality"}, {}, {"mortality": 1})

psepsis_patient = CTF({sepsis}, {patient}, name="Probability")

compute_ctf(ncm, psepsis_patient, n=100_000, evaluating=True)

0.18939319252967834

In [113]:
compute_ctf(ncm, CTF({dead}, {patient, sepsis}), n=100_000, evaluating=True)

0.5378521084785461

In [114]:
def print_sepsis_mortality_totals(df, label=None):
    if label: print(f"--{label}--")
    print(f"Total: {len(df)}")
    print(f"Sepsis: {sum(df['sepsis'])} ({sum(df['sepsis'])*100/len(df):.2f}%)")
    print(f"Mortality: {sum(df['mortality'])} ({sum(df['mortality'])*100/len(df):.2f}%)\n")

def filter_df(df): return df[(df['age']==0) & (df['gender']==1) & (df['ethnicity']==1) & (df['sepsis']==1)]
train_demo_df = filter_df(train_df)
test_demo_df = filter_df(test_df)
total_demo_df = filter_df(processed_dt.data)

print_sepsis_mortality_totals(train_demo_df, 'Train DF')
print_sepsis_mortality_totals(test_demo_df, 'Test DF')
print_sepsis_mortality_totals(total_demo_df, 'Total Processed DF')

--Train DF--
Total: 1166
Sepsis: 1166.0 (100.00%)
Mortality: 197.0 (16.90%)

--Test DF--
Total: 150
Sepsis: 150.0 (100.00%)
Mortality: 24.0 (16.00%)

--Total Processed DF--
Total: 1316
Sepsis: 1316.0 (100.00%)
Mortality: 221.0 (16.79%)



In [124]:
print_sepsis_mortality_totals(train_df, 'Train DF')
print_sepsis_mortality_totals(test_df, 'Test DF')
print_sepsis_mortality_totals(processed_dt.data, 'Total Processed DF')

--Train DF--
Total: 117519
Sepsis: 17001.0 (14.47%)
Mortality: 9194.0 (7.82%)

--Test DF--
Total: 13166
Sepsis: 1962.0 (14.90%)
Mortality: 1032.0 (7.84%)

--Total Processed DF--
Total: 130685
Sepsis: 18963.0 (14.51%)
Mortality: 10226.0 (7.82%)



In [125]:
compute_ctf(ncm, CTF({dead}), n=100_000, evaluating=True)

0.09554000198841095

In [126]:
compute_ctf(ncm, CTF({sepsis}), n=100_000, evaluating=True)

0.10995999723672867

In [129]:
from src.helpers import *
# get_df("patient")["age"].describe()
eicu.data.describe()

,hospitalid,age,mortality,sepsis,ventilation,intravenous_fluid,bp,sao2,heartrate,respiration
count,130685.000000,130685.000000,130681.000000,130685.000000,130685.000000,130685.000000,128612.000000,127580.000000,128356.000000,120751.000000
mean,285.199633,63.117856,0.078252,0.145105,0.080399,0.073842,81.822273,96.433952,83.675468,19.476265
std,105.495387,17.430184,0.268568,0.352208,0.271912,0.261514,12.666260,2.848294,14.789047,4.418594
min,79.000000,0.000000,0.000000,0.000000,0.000000,0.000000,23.333333,5.739777,0.000000,0.000000
25%,197.000000,53.000000,0.000000,0.000000,0.000000,0.000000,72.888889,95.401367,73.150002,16.556551
50%,272.000000,65.000000,0.000000,0.000000,0.000000,0.000000,80.600000,96.741402,82.897247,18.888889
75%,391.000000,76.000000,0.000000,0.000000,0.000000,0.000000,89.643954,97.988968,93.357666,21.767700
max,459.000000,100.000000,1.000000,1.000000,1.000000,1.000000,158.654762,100.000000,179.333328,148.778931


mostly working on full stack MyChart (C# and react)
ML evaluating conversations & scoring them
having AI generate HTML and embed directly into MyChart

In [119]:
eicu_pat = data_age_cut[(data_age_cut['age']=='old') & (data_age_cut['gender']=='Male') & (data_age_cut['ethnicity']=='POC/Other')]
eicu_pat[eicu_pat['sepsis']==1]

,uniquepid,hospitalid,age,gender,ethnicity,mortality,sepsis,ventilation,intravenous_fluid,bp,sao2,heartrate,respiration
50,003-11096,93,old,Male,POC/Other,0.0,1,1.0,1.0,NaN,NaN,NaN,NaN
105,003-12232,86,old,Male,POC/Other,0.0,1,1.0,0.0,58.560000,98.444443,70.800003,30.581818
181,003-13962,84,old,Male,POC/Other,1.0,1,1.0,1.0,63.240369,95.658295,106.489716,16.777779
274,003-16483,79,old,Male,POC/Other,0.0,1,1.0,1.0,76.855362,96.670189,84.693886,21.884287
347,003-18294,84,old,Male,POC/Other,0.0,1,1.0,0.0,103.747368,91.324326,118.236839,31.639999
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129728,035-423,458,old,Male,POC/Other,0.0,1,0.0,1.0,102.500000,99.133331,87.990479,19.457144
129897,035-5166,458,old,Male,POC/Other,0.0,1,1.0,0.0,83.111111,96.214676,79.026184,20.900141
130021,035-5823,459,old,Male,POC/Other,0.0,1,1.0,1.0,70.386326,97.248222,97.854362,29.963905
130126,035-6439,458,old,Male,POC/Other,1.0,1,1.0,0.0,83.340206,97.029274,84.867165,19.063423


In [5]:
from src.helpers import * 

patient = get_df('patient')
patient['unitstaytype'].value_counts()

unitstaytype
admit             154948
stepdown/other     25239
transfer           10725
readmit             9947
Name: count, dtype: int64

In [2]:
from src.preprocess import *
preprocess_admissionDx()

,patientunitstayid,admissionDx_cat,admissionDx_str,operative,or_within_4hrs,non_operative_organs,operative_organs,elective
0,2900217,Cardiovascular,Aortic and Mitral valve replacement,True,True,NaN,Cardiovascular,Yes
1,2900240,Gastrointestinal,"Bleeding, GI-location unknown",False,False,Gastrointestinal,NaN,NaN
2,2900262,Cardiovascular,"Infarction, acute myocardial (MI)",False,False,Cardiovascular,NaN,NaN
3,2900292,Genitourinary,"Renal failure, acute",False,False,Genitourinary,NaN,NaN
4,2900317,Respiratory,Emphysema/bronchitis,False,False,Respiratory,NaN,NaN
...,...,...,...,...,...,...,...,...
177858,2900161,Cardiovascular,Rhythm disturbance (conduction defect),False,False,Cardiovascular,NaN,NaN
177859,2900171,Musculoskeletal/Skin,Cellulitis and localized soft tissue infections,False,False,Musculoskeletal/Skin,NaN,NaN
177860,2900182,Metabolic/Endocrine,Hypoglycemia,False,False,Metabolic/Endocrine,NaN,NaN
177861,2900186,Cardiovascular,"Sepsis, unknown",False,False,Cardiovascular,NaN,NaN


In [ ]:

eicu: eICUData = eICUData.load(datapath)

treat_dict = {V: 1 for V in eicu.treatments}
notreat_dict = {V: 0 for V in eicu.treatments}

Y = "mortality"

y1dox1 = CTFTerm({Y}, treat_dict, {'Y': 1})
y1dox0 = CTFTerm({Y}, notreat_dict, {'Y': 1})
y0dox1 = CTFTerm({Y}, treat_dict, {'Y': 0})
y0dox0 = CTFTerm({Y}, notreat_dict, {'Y': 0})
x1 = CTFTerm(eicu.treatments, {}, treat_dict)

py0dox0_x1 = CTF({y0dox0}, {x1}, name="ETT")
py1dox0_x1 = CTF({y1dox0}, {x1}, name="ETT")
py0dox1_x1 = CTF({y0dox1}, {x1}, name="ETT")
py1dox1_x1 = CTF({y1dox1}, {x1}, name="ETT")